# Colab 42 — architecture ablations for Chapter 5

**SNNEED only.** Every arm is the deployed model with exactly one thing changed, evaluated on
Spearman (overall and by range), AUROC, MAP@10 and high-range RMSE, on all four datasets.

| Arm | Question | Configurations |
|---|---|---|
| **0** | anchor | `K=16`, pooling, 30,000 pairs, 30 epochs — the deployed model |
| **P** | does the pooling width matter? | `K = 8, 16, 32` |
| **N** | does pooling matter at all? | pooling replaced by a flatten over the full padded width |
| **T** | how much training data is needed? | 5,000 / 10,000 / 30,000 / 100,000 pairs, 30 epochs |
| **T2** | …with the optimiser budget held fixed | the same sizes at matched optimiser steps *(optional)* |
| **E** | how long does it need to train? | 5 / 10 / 20 / 30 / 50 epochs |

### Four things to read before running

**1. Pooling here is always padded-width.** Sequences are padded to a fixed width of 200 and
`AdaptiveAvgPool1d(K)` splits *that* axis into K windows, so window boundaries sit at fixed absolute
positions and short sequences leave trailing windows structurally zero. That is how sequence length
survives into the embedding, and the length-constraint run found it load-bearing. Arm P varies K
within this design. **Do not "improve" the padding** — pooling over each sequence's true length is a
*different model*, already measured and already worse on every natural dataset.

**2. The arms are isolated and resumable.** Each arm is its own cell and can be run alone. A
configuration is written to a ledger on Drive as soon as all four of its datasets are evaluated, and
re-running any cell skips completed work. Arm E writes after *each* epoch checkpoint, so a disconnect
mid-sweep keeps everything up to the last checkpoint. Saved encoder checkpoints are for later
inspection — **they are not used to resume training**, so an interrupted training run does restart.

**3. The epoch arm does not retrain.** A model trained for 30 epochs *is* the 30-epoch checkpoint of
a 50-epoch run — there is no early stopping, no validation split and no learning-rate schedule, so
nothing depends on the total budget. Arm E trains once to 50 epochs per seed and evaluates at each
checkpoint.

**4. ⚠ The no-pooling arm confounds pooling with capacity.** Flattening the full padded width feeds
`64 x 200 = 12,800` values into the projection instead of `64 x 16 = 1,024`, so the model goes from
**141,184 to 1,648,512 parameters** — 11.7x. That is the literal ablation and it is what the earlier
run measured, but any difference it shows has two possible causes. The parameter count is recorded in
every row so the confound stays visible. Arm P is the milder lever.

### ⚠ Two things this notebook cannot claim

**`K=32` here is not a replication of the earlier 272,256-parameter result.** That configuration used
**true-length** pooling; this one is padded-width. The parameter counts coincide because parameters
depend only on `K` and the channel width, never on which axis is pooled over. Agreement would not
reproduce the earlier finding and disagreement would not overturn it. Padded-width `K` has never been
swept before, so arm P is a new measurement, not a confirmation.

**Arm T varies data and optimiser steps together.** At a fixed 30 epochs, 5,000 pairs gets ~1,170
optimiser steps and 100,000 pairs gets ~23,460. Arm T therefore measures *training-set size under a
fixed epoch budget*, which is a real and reportable quantity but is **not** an isolated data-size
effect. Arm T2 repeats the sweep with the step count held at the baseline's, and is **the arm to
quote if the claim is about data**. It is enabled (`RUN_MATCHED_STEPS = True`); set it to `False` to
skip it and save about 75 minutes.

### Runtime

Rough T4 estimate, 3 seeds throughout. **Arm T dominates because of the 100,000-pair configuration.**

| Arm | Trainings | Estimate |
|---|---|---|
| 0 | 3 | ~25 min |
| P | 6 | ~50 min |
| N | 3 | ~30 min |
| T | 9 | ~90 min |
| T2 | 9 | ~75 min *(optional)* |
| E | 3 (to 50 epochs) | ~40 min |

If the session will not hold, run the arms in separate sessions — the ledger makes that free. To trim,
drop `100_000` from `TRAIN_GRID` first; it costs more than the rest of arm T combined.

### Outputs

`colab42_ablations_raw.csv` (one row per arm x configuration x seed x dataset),
`colab42_ablations_mean.csv` (seed means and standard deviations), `colab42_ablations.json`, and one
printed table per arm. **The output cell refuses to write an incomplete experiment** unless
`ALLOW_PARTIAL` is set.

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')
!pip install rapidfuzz --quiet

## Constants

Identical to colab40 down to the seeds, plus the ablation grids. Nothing here may drift from the run
of record — if it does, the baseline arm stops being a check.

`PROTOCOL_SIG` fingerprints the constants that affect a result. It is written into every ledger row,
and rows whose signature differs from the current one are **not** counted as completed work, so
results produced by an earlier version of this notebook can never merge silently into a new run.

In [ ]:
import os, json, time, pickle, copy, math, subprocess, platform, hashlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

DATA_DIR = 'sampledata/cath'

# --- IDENTICAL to the run of record (colab40) ------------------------------
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
RANGE_LOW, RANGE_HIGH = 0.30, 0.70
RESCUED = {'4z0mC02', '3qkaE02'}
N_TRAIN, EPOCHS, LR = 30_000, 30, 1e-3
SEEDS = [0, 1, 2]
STRAT_CAND, STRAT_PER_BIN = 200_000, 400
PAIR_SEED = 999
SYN_PERTURB, SYN_INDEP, SYN_SEED = 20_000, 8_000, 20260810

DATASETS = ['Synth', '3Di', 'SS', 'AA']

# --- the ablation grids ----------------------------------------------------
POOL_GRID  = [8, 16, 32]                              # arm P
TRAIN_GRID = [5_000, 10_000, 30_000, 100_000]         # arms T and T2
EPOCH_GRID = [5, 10, 20, 30, 50]                      # arm E, one training run

RUN_MATCHED_STEPS = True    # arm T2. On: any claim about data volume rests on this arm, not arm T.
ALLOW_PARTIAL     = False   # let the output cell write an incomplete experiment

# --- the Chapter 4 SNNEED row, for the arm-0 check -------------------------
CH4 = {'spearman':  {'Synth': 0.925, '3Di': 0.950, 'SS': 0.961, 'AA': 0.204},
       'auroc':     {'Synth': 0.969, '3Di': 0.992, 'SS': 0.986, 'AA': 1.000},
       'map10':     {'Synth': 0.977, '3Di': 0.502, 'SS': 0.405, 'AA': 0.928},
       'rmse_high': {'Synth': 0.113, '3Di': 0.070, 'SS': 0.054, 'AA': 0.108}}

# --- the run of record's evaluation-set sizes, asserted below --------------
REF_SIZES = {'Synth': dict(pairs=3_648, high=1_205, queries=2_410),
             '3Di':   dict(pairs=3_699, high=1_225, queries=347),
             'SS':    dict(pairs=4_000, high=1_403, queries=10_002),
             'AA':    dict(pairs=1_216, high=5,     queries=10)}
REF_COLL = {'Synth': 7_296, '3Di': 10_501, 'SS': 10_497, 'AA': 10_501}
REF_POS  = {'Synth': 1_205, '3Di': 6_009, 'SS': 623_077, 'AA': 5}

PROTOCOL_SIG = (f'v2|K{K}|len{MIN_LEN}-{MAX_LEN}|bs{BS}|lr{LR}|ntrain{N_TRAIN}|ep{EPOCHS}'
                f'|pair{PAIR_SEED}|syn{SYN_SEED}|strat{STRAT_CAND}x{STRAT_PER_BIN}'
                f'|rng{RANGE_LOW}-{RANGE_HIGH}')
print('protocol signature:', PROTOCOL_SIG)

def steps_per_epoch(n): return math.ceil(n / BS)
BASE_STEPS = steps_per_epoch(N_TRAIN) * EPOCHS
def matched_epochs(n): return max(1, round(BASE_STEPS / steps_per_epoch(n)))
print(f'baseline optimiser steps: {BASE_STEPS:,}')

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

# --- Drive cache -----------------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/thesis_artefacts'
except Exception as e:
    CACHE = '/content/thesis_artefacts'
    print('Drive not mounted, caching locally (will NOT survive the session):', e)
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{CACHE}/colab42_ckpt', exist_ok=True)
print('artefact cache:', CACHE)

## Provenance

Chapter 3's environment table has to describe *these* ablations as well as the master run, and Colab
changes its images without warning. The commit is recorded too, so a result can be traced to the code
that produced it.

In [ ]:
def _git(*args):
    try: return subprocess.check_output(['git', *args], text=True).strip()
    except Exception: return 'unavailable'

ENV = dict(python=platform.python_version(), torch=torch.__version__,
           numpy=np.__version__, pandas=pd.__version__,
           cuda=torch.version.cuda, gpu=(torch.cuda.get_device_name(0)
                                         if torch.cuda.is_available() else 'cpu'),
           commit=_git('rev-parse', 'HEAD'), branch=_git('rev-parse', '--abbrev-ref', 'HEAD'),
           dirty=bool(_git('status', '--porcelain')), when=time.strftime('%Y-%m-%d %H:%M:%S'),
           protocol_sig=PROTOCOL_SIG)
try:
    import rapidfuzz, sklearn, scipy
    ENV.update(rapidfuzz=rapidfuzz.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__)
except Exception: pass
print(json.dumps(ENV, indent=2))

## Stage A — evaluation objects, reused from colab40

`relevance_sets.pkl` and `balanced_pairs.pkl` are the *same objects Chapter 4 was measured on*. If
they are missing from Drive the notebook rebuilds them, which costs about an hour on SS.

The cached pair sets are **validated, not merely counted**. Counting alone would pass a stale cache
built from a different sampling run, so a random subsample of every balanced pair set has its
normalised Levenshtein similarity recomputed from the collections and compared against the stored
value. If the pickle and the collections disagree, that check fires.

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

COLL = {
    'AA':  [s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)],
    'SS':  [s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)],
    '3Di': [s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)],
}
for f in ['AA', 'SS', '3Di']:
    assert len(COLL[f]) == REF_COLL[f], f'{f} collection differs from the run of record - STOP'
print('collections match Section 3.4')

In [ ]:
def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN + 1)); return ''.join(rng.choice(list(abc), size=L))

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0:            op = 'ins'
        elif len(s) >= MAX_LEN:    op = rng.choice(['sub', 'del'])
        else:                      op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def build_synth_eval(seed=SYN_SEED):
    r = np.random.default_rng(seed); recs = []
    for _ in range(SYN_PERTURB):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base) + 1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(SYN_INDEP):
        recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]
    nl_all = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl_all, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:STRAT_PER_BIN].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, l = recs[int(idx)]
        I.append(len(seqs)); seqs.append(a)
        J.append(len(seqs)); seqs.append(b)
        NL.append(l)
    return seqs, np.array(I), np.array(J), np.array(NL)

print('regenerating the Synth evaluation collection - about a minute...')
COLL['Synth'], SYN_I, SYN_J, SYN_NL = build_synth_eval()
assert len(COLL['Synth']) == REF_COLL['Synth'], 'Synth collection differs - STOP'
assert len(SYN_NL) == REF_SIZES['Synth']['pairs'], 'Synth balanced pair count differs - STOP'
print(f'  {len(COLL["Synth"]):,} sequences, {len(SYN_NL):,} balanced pairs - matches colab40')

In [ ]:
REL_PATH   = f'{CACHE}/relevance_sets.pkl'
STRAT_PATH = f'{CACHE}/balanced_pairs.pkl'

def build_relevance(seqs, block=1024, tag=''):
    N = len(seqs); lens = np.array([len(s) for s in seqs])
    T_high, pos = {}, []
    t0 = time.time()
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - D / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= RANGE_HIGH)[0]
            if hi.size:
                T_high[i] = hi.astype(np.int32)
                for j in hi:
                    if j > i: pos.append((i, int(j), float(row[j])))
        print(f'    {tag} {r1:>6,}/{N:,}  ({time.time()-t0:.0f}s)', end='\r')
    print()
    return dict(T_high=T_high, pos_pairs=pos)

def build_balanced(feed, rng):
    seqs = COLL[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if REL[feed]['pos_pairs']:
        pa = np.array(REL[feed]['pos_pairs'], dtype=float)
        a  = np.concatenate([a, pa[:, 0].astype(np.int64)])
        b  = np.concatenate([b, pa[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, pa[:, 2]])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9)
    take, supply = [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]; supply.append(int(idx.size))
        if idx.size: take.extend(rng.permutation(idx)[:STRAT_PER_BIN].tolist())
    take = np.array(take, dtype=np.int64)
    return dict(i=a[take], j=b[take], nl=nl[take], supply=supply)

if os.path.exists(REL_PATH):
    with open(REL_PATH, 'rb') as fh: REL = pickle.load(fh)
    print('loaded cached relevance sets from Drive')
else:
    print('⚠ relevance sets not cached - rebuilding, SS takes about an hour')
    REL = {f: build_relevance(COLL[f], tag=f) for f in DATASETS}
    with open(REL_PATH, 'wb') as fh: pickle.dump(REL, fh)

if os.path.exists(STRAT_PATH):
    with open(STRAT_PATH, 'rb') as fh: STRAT = pickle.load(fh)
    print('loaded cached balanced pair sets from Drive')
else:
    rng = np.random.default_rng(PAIR_SEED)
    STRAT = {f: build_balanced(f, rng) for f in ['AA', 'SS', '3Di']}
    with open(STRAT_PATH, 'wb') as fh: pickle.dump(STRAT, fh)
STRAT['Synth'] = dict(i=SYN_I, j=SYN_J, nl=SYN_NL, supply=None)

In [ ]:
# ---- exact counts -------------------------------------------------------
for f in DATASETS:
    assert len(REL[f]['pos_pairs']) == REF_POS[f], \
        f'{f} high-similarity pair count is {len(REL[f]["pos_pairs"]):,}, expected {REF_POS[f]:,} - STOP'
    assert len(REL[f]['T_high']) == REF_SIZES[f]['queries'], \
        f'{f} eligible-query count differs from Table 3.8 - STOP'
    nl = STRAT[f]['nl']
    assert len(nl) == REF_SIZES[f]['pairs'], \
        f'{f} balanced pair count is {len(nl):,}, expected {REF_SIZES[f]["pairs"]:,} - STOP'
    assert int((nl >= RANGE_HIGH).sum()) == REF_SIZES[f]['high'], \
        f'{f} high-range pair count is {int((nl >= RANGE_HIGH).sum()):,}, ' \
        f'expected {REF_SIZES[f]["high"]:,} - STOP'

# ---- the cache actually matches the collections -------------------------
# Counts alone would pass a stale pickle sampled from a different run. Recompute the
# target for a random subsample of every pair set and compare against what was stored.
_chk = np.random.default_rng(20260828)
for f in DATASETS:
    P = STRAT[f]; n = len(P['nl'])
    assert P['i'].max() < len(COLL[f]) and P['j'].max() < len(COLL[f]), \
        f'{f} cached pair indices fall outside the collection - STOP'
    sel = _chk.choice(n, size=min(200, n), replace=False)
    got = np.array([norm_lev(COLL[f][int(P['i'][s])], COLL[f][int(P['j'][s])]) for s in sel])
    bad = int((np.abs(got - P['nl'][sel]) > 1e-9).sum())
    assert bad == 0, (f'{f}: {bad}/{len(sel)} cached pairs do NOT reproduce their stored '
                      f'similarity from the collections. The cache is stale - DELETE '
                      f'{STRAT_PATH} AND {REL_PATH} AND RERUN.')
    print(f'  {f:<6} {n:>6,} pairs   {int((P["nl"] >= RANGE_HIGH).sum()):>6,} high   '
          f'{len(REL[f]["T_high"]):>7,} queries   subsample check passed')

FINGERPRINT = hashlib.sha256(
    b''.join(np.ascontiguousarray(STRAT[f][k]).tobytes()
             for f in DATASETS for k in ('i', 'j', 'nl'))).hexdigest()[:16]
print(f'\nevaluation objects validated - fingerprint {FINGERPRINT}')
print('these are the objects Chapter 4 was measured on')

## Metrics

Copied verbatim from colab40. `map10_untied` is the one SNNEED uses there — the chord readout is
continuous, so the tie-resampling path of Section 3.6.5 is not needed.

In [ ]:
RANGES = {'far':  lambda nl: nl < RANGE_LOW,
          'mid':  lambda nl: (nl >= RANGE_LOW) & (nl < RANGE_HIGH),
          'high': lambda nl: nl >= RANGE_HIGH}
MIN_N_RANGE = 10

# AA's mid range holds 11 pairs and its high range 5. Both are below anything that may be
# reported, and 11 clears MIN_N_RANGE, so it is suppressed HERE rather than in the prose -
# a number that is never written cannot be quoted by accident.
SUPPRESS_RANGE = {('AA', 'mid'), ('AA', 'high')}

def rho(sim, nl):
    if len(nl) < MIN_N_RANGE or np.ptp(nl) == 0: return np.nan
    r = spearmanr(sim, nl).correlation
    return float(r) if r == r else np.nan

def auroc(sim, nl):
    y = (nl >= RANGE_HIGH).astype(int)
    return float(roc_auc_score(y, sim)) if 0 < y.sum() < len(y) else np.nan

def rmse_high(pred, nl):
    m = nl >= RANGE_HIGH
    return float(np.sqrt(np.mean((pred[m] - nl[m]) ** 2))) if m.sum() else np.nan

def _ap(rel_seq, R, k=10):
    c = np.cumsum(rel_seq); prec = c / np.arange(1, k + 1)
    return float((rel_seq * prec).sum() / min(R, k))

def map10_untied(score_rows, T_high, k=10):
    aps = []
    for qi, rel in T_high.items():
        s = score_rows(qi); s[qi] = -np.inf
        top = np.argpartition(-s, k)[:k]; top = top[np.argsort(-s[top])]
        ts = set(rel.tolist())
        aps.append(_ap(np.array([1 if o in ts else 0 for o in top]), len(rel), k))
    return float(np.mean(aps)) if aps else np.nan

print('metrics defined; suppressed range coefficients:', sorted(SUPPRESS_RANGE))

## The encoder and its variants

`EncPool(k)` at `k=16` is the deployed encoder, character for character. `EncFlat` is the no-pooling
arm: the same masked convolutional map, flattened at the full padded width.

⚠ **Pooling is over the padded width in every variant.** Padded positions contribute zeros to their
window, which is how sequence length survives into the embedding. Pooling over each sequence's true
length is a different model and is not what this notebook varies.

In [ ]:
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX] * (MAX_LEN - len(idx))
    return torch.tensor(idx, dtype=torch.long)

class EncPool(nn.Module):
    '''Deployed encoder. k=16 reproduces the run of record exactly.'''
    def __init__(s, k=K):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(k); s.fc = nn.Linear(64 * k, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)

class EncFlat(nn.Module):
    '''No pooling: flatten the masked convolutional map at full padded width.
       ⚠ 1,648,512 parameters against the deployed 141,184 - capacity is confounded.'''
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.fc = nn.Linear(64 * MAX_LEN, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(h.flatten(1)), p=2, dim=1)

class RegModel(nn.Module):
    '''Deployed SNNEED: chord readout, no head. The readout has no parameters.'''
    def __init__(s, enc): super().__init__(); s.encoder = enc
    def forward(s, a, b):
        ea, eb = s.encoder(a), s.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0

class DS(Dataset):
    def __init__(s, pp): s.p = pp
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), torch.tensor(l, dtype=torch.float32)

def build_train_pairs(n, seed):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1 - t) * L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

for k in POOL_GRID:
    print(f'  K={k:<3} {sum(p.numel() for p in EncPool(k).parameters()):>10,} parameters')
print(f'  flatten {sum(p.numel() for p in EncFlat().parameters()):>9,} parameters')

## Training and evaluation

`train_variant` trains one configuration and returns a snapshot at every requested epoch checkpoint.
That is what makes arm E cost one training run instead of five: with no early stopping, no validation
split and no learning-rate schedule, the state after 30 epochs of a 50-epoch run *is* the 30-epoch
model.

Two details that matter for reading the arms:

**Minibatch order is independent of the architecture.** The DataLoader draws from its own generator
seeded from `seed` alone. Without that, a larger projection consumes a different amount of the global
random state during initialisation and every variant would see a *different batch order* as well as a
different architecture.

**`train_mse_final` is measured, not accumulated.** It is a separate no-grad pass over the training
pairs with the finished checkpoint. `train_loss_online` — the running average over the last epoch,
taken while the parameters were still moving — is kept alongside for reference, but only the former
can support a statement about fitting the training set.

`evaluate` uses the same scores the run of record uses: the cosine of the unit embeddings for the rank
metrics, and the chord readout for RMSE. They induce the same ordering by construction.

In [ ]:
@torch.no_grad()
def training_mse(model, pairs, bs=512):
    '''True training error of the finished model: one no-grad pass, parameters fixed.'''
    model.eval(); tot = 0.0
    for i in range(0, len(pairs), bs):
        chunk = pairs[i:i+bs]
        xa = torch.stack([encode_pad(a) for a, _, _ in chunk]).to(device)
        xb = torch.stack([encode_pad(b) for _, b, _ in chunk]).to(device)
        y  = torch.tensor([l for _, _, l in chunk], dtype=torch.float32, device=device)
        tot += float(((model(xa, xb) - y) ** 2).sum())
    return tot / len(pairs)

def train_variant(make_enc, n_train, seed, eval_at):
    '''Train once, snapshot at each epoch in eval_at.
       Returns {epoch: (model, online_loss, final_mse, steps)}.'''
    eval_at = sorted(set(eval_at))
    pairs = build_train_pairs(n_train, seed)
    torch.manual_seed(seed)
    model = RegModel(make_enc()).to(device)
    g = torch.Generator(); g.manual_seed(10_000 + seed)   # batch order, architecture-independent
    dl = DataLoader(DS(pairs), batch_size=BS, shuffle=True, generator=g)
    opt = torch.optim.Adam(model.parameters(), LR)
    snaps = {}; steps = 0; t0 = time.time()
    for ep in range(max(eval_at)):
        model.train(); tot = 0.0
        for xa, xb, y in dl:
            xa, xb, y = xa.to(device), xb.to(device), y.to(device)
            loss = F.mse_loss(model(xa, xb), y)          # UNWEIGHTED - Section 3.3.2
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(y); steps += 1
        online = tot / len(pairs)
        if (ep + 1) in eval_at:
            snap = copy.deepcopy(model).eval()
            snaps[ep + 1] = (snap, float(online), training_mse(snap, pairs), steps)
            print(f'      checkpoint epoch {ep+1:>3}  online {online:.5f}  '
                  f'final {snaps[ep+1][2]:.5f}  {steps:,} steps  ({time.time()-t0:.0f}s)')
        elif (ep + 1) % 10 == 0:
            print(f'      epoch {ep+1:>3}/{max(eval_at)}  online {online:.5f}  '
                  f'({time.time()-t0:.0f}s)')
    return snaps

@torch.no_grad()
def embed(model, feed, bs=256):
    out = []
    for i in range(0, len(COLL[feed]), bs):
        x = torch.stack([encode_pad(s) for s in COLL[feed][i:i+bs]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def evaluate(model, feed):
    E = embed(model, feed); P = STRAT[feed]; nl = P['nl']
    sim  = np.sum(E[P['i']] * E[P['j']], axis=1)                       # cosine
    pred = 1.0 - np.linalg.norm(E[P['i']] - E[P['j']], axis=1) / 2.0   # chord readout
    Et = torch.as_tensor(E, device=device)
    rows = lambda qi: (Et[qi] @ Et.t()).cpu().numpy().astype(np.float64)
    rec = dict(dataset=feed, spearman=rho(sim, nl), auroc=auroc(sim, nl),
               map10=map10_untied(rows, REL[feed]['T_high']),
               rmse_high=rmse_high(pred, nl),
               n_pairs=int(len(nl)), n_queries=len(REL[feed]['T_high']))
    for name, sel in RANGES.items():
        m = sel(nl)
        if (feed, name) in SUPPRESS_RANGE:
            rec[f'spearman_{name}'] = np.nan          # never estimated, never persisted
        else:
            rec[f'spearman_{name}'] = rho(sim[m], nl[m]) if m.sum() >= MIN_N_RANGE else np.nan
        rec[f'n_{name}'] = int(m.sum())
    return rec

print('training and evaluation defined')

## The ledger

A configuration counts as complete only when **all four datasets** are present for it *and* its
protocol signature matches the current one. Rows written by an earlier version of this notebook do
not count as done, so results from different code can never merge into one table.

Arm E writes after each epoch checkpoint, so an interrupted sweep keeps every checkpoint already
evaluated. Encoder checkpoints are saved for later inspection; **they are not used to resume
training**, so an interrupted training run does start over.

To force a re-run of one configuration, delete its rows from the ledger. To start over, delete the
file.

In [ ]:
LEDGER = f'{CACHE}/colab42_ledger.csv'
LED = (pd.read_csv(LEDGER) if os.path.exists(LEDGER)
       else pd.DataFrame(columns=['arm', 'config', 'seed', 'dataset', 'protocol_sig']))

if len(LED) and 'protocol_sig' in LED.columns:
    other = sorted(set(LED['protocol_sig'].dropna()) - {PROTOCOL_SIG})
    if other:
        print('⚠ the ledger contains rows from a DIFFERENT protocol signature:')
        for s in other: print('   ', s, f'({int((LED["protocol_sig"] == s).sum())} rows)')
        print('  they will NOT be counted as completed work and will NOT be written to the outputs.')

def _current():
    if len(LED) == 0: return LED
    return LED[LED['protocol_sig'] == PROTOCOL_SIG] if 'protocol_sig' in LED.columns else LED.iloc[0:0]

def is_done(arm, config, seed):
    d = _current()
    if len(d) == 0: return False
    m = (d['arm'] == arm) & (d['config'] == config) & (d['seed'] == seed)
    return int(d[m]['dataset'].nunique()) == len(DATASETS)      # partial rows do not count

def push(rows):
    global LED
    for r in rows: r['protocol_sig'] = PROTOCOL_SIG
    LED = pd.concat([LED, pd.DataFrame(rows)], ignore_index=True)
    LED.to_csv(LEDGER, index=False)

def table(arms, metric, order=None, index=None):
    d = _current()
    d = d[d['arm'].isin(arms)] if len(d) else d
    if not len(d): print('  nothing in the ledger yet for', arms); return
    t = d.pivot_table(index='config', columns='dataset', values=metric, aggfunc='mean')
    if order is not None: t = t.reindex(order)
    if index is not None: t.index = index
    print(f'\n{metric} (seed means)')
    print(t[[c for c in DATASETS if c in t.columns]].round(3).to_string())

def run_config(arm, make_enc, n_train, seed, labels, tag=''):
    '''labels: {epoch: config_label}. Trains once, evaluates and WRITES each checkpoint.'''
    todo = {e: lab for e, lab in labels.items() if not is_done(arm, lab, seed)}
    if not todo:
        print(f'  [skip] {arm} {tag} seed {seed} - already complete in the ledger'); return
    print(f'  {arm} {tag} seed {seed}')
    n_params = sum(p.numel() for p in make_enc().parameters())
    snaps = train_variant(make_enc, n_train, seed, list(todo))
    for e in sorted(todo):
        lab = todo[e]; model, online, final_mse, steps = snaps[e]
        safe = f'{arm}_{lab}_s{seed}'.replace('=', '').replace(',', '').replace(' ', '')
        torch.save(model.encoder.state_dict(), f'{CACHE}/colab42_ckpt/{safe}.pt')
        rows = []
        for f in DATASETS:
            r = evaluate(model, f)
            r.update(arm=arm, config=lab, seed=seed, n_train=n_train, epochs=e,
                     n_steps=steps, n_params=n_params,
                     train_loss_online=online, train_mse_final=final_mse)
            rows.append(r)
            print(f'    {lab:<16} {f:<6} rho={r["spearman"]:+.3f} AUROC={r["auroc"]:.3f} '
                  f'MAP@10={r["map10"]:.3f} RMSE_high={r["rmse_high"]:.3f}')
        push(rows)          # written per checkpoint, so a disconnect keeps what is done

print(f'ledger: {LEDGER}   {len(LED)} rows on file, '
      f'{len(_current())} at the current protocol signature')

## Arm 0 — baseline

`K=16`, pooling, 30,000 pairs, 30 epochs. This *is* the deployed model, so it should reproduce the
Chapter 4 SNNEED row.

⚠ **It will not reproduce it exactly, and that is expected** — SNNEED training is not bit-reproducible
across GPU sessions even with fixed seeds. The 25 and 26 August master runs differ by up to 0.03 on
the same code. The comparison printed below is a sanity check, not an assert. If anything moves by
more than about 0.05, stop and find out why before running the other arms.

In [ ]:
for seed in SEEDS:
    run_config('0-baseline', lambda: EncPool(16), N_TRAIN, seed,
               {EPOCHS: 'K=16'}, tag='K=16')

b = _current(); b = b[b['arm'] == '0-baseline']
print(f'\n{"metric":<11}{"dataset":<8}{"colab42":>9}{"ch.4":>8}{"delta":>8}')
worst = 0.0
for m, ref in CH4.items():
    for f in DATASETS:
        v = b[b['dataset'] == f][m].mean()
        d = v - ref[f]; worst = max(worst, abs(d))
        flag = '  <-- CHECK' if abs(d) > 0.05 else ''
        print(f'{m:<11}{f:<8}{v:>9.3f}{ref[f]:>8.3f}{d:>+8.3f}{flag}')
print(f'\nlargest absolute difference from Chapter 4: {worst:.3f}')
print('under 0.05 is seed noise; above it, stop and investigate')

## Arm P — pooling width

`K = 8, 16, 32`, padded-width throughout. `K=16` is taken from arm 0 and is not retrained.

⚠ **This is a new measurement, not a confirmation of anything.** The earlier 272,256-parameter
finding used **true-length** pooling; `K=32` here is padded-width. The parameter counts coincide
because parameters depend only on `K` and the channel width, not on the axis pooled over. Padded-width
`K` has never been swept, so agreement with the earlier result would not reproduce it and disagreement
would not overturn it.

In [ ]:
for k in POOL_GRID:
    if k == 16: continue                       # arm 0 already has it
    for seed in SEEDS:
        run_config('P-pooling-width', lambda k=k: EncPool(k), N_TRAIN, seed,
                   {EPOCHS: f'K={k}'}, tag=f'K={k}')

print('=== Arm P - pooling width (padded) ===')
for m in ['spearman', 'spearman_high', 'auroc', 'map10', 'rmse_high']:
    table(['0-baseline', 'P-pooling-width'], m, order=[f'K={k}' for k in POOL_GRID])

d = _current(); d = d[d['arm'].isin(['0-baseline', 'P-pooling-width'])]
print('\nparameters, optimiser steps and training error')
print(d.groupby('config')[['n_params', 'n_steps', 'train_mse_final']].mean().round(5).to_string())

## Arm N — pooling against no pooling

⚠ Read this arm with the parameter count in view. `EncFlat` has **1,648,512 parameters against the
deployed 141,184**. Whatever it shows has two candidate explanations, and this arm alone cannot
separate them. Arm P varies the pooling width far more mildly.

In [ ]:
for seed in SEEDS:
    run_config('N-no-pooling', EncFlat, N_TRAIN, seed, {EPOCHS: 'flatten'}, tag='flatten')

print('=== Arm N - pooling against no pooling ===')
for m in ['spearman', 'spearman_high', 'auroc', 'map10', 'rmse_high']:
    table(['0-baseline', 'N-no-pooling'], m, order=['K=16', 'flatten'])

p = _current(); p = p[p['arm'].isin(['0-baseline', 'N-no-pooling'])]
print('\nparameters and training error')
print(p.groupby('config')[['n_params', 'train_mse_final']].mean().round(5).to_string())

## Arm T — training set size at a fixed epoch budget

5,000 / 10,000 / 30,000 / 100,000 pairs, epochs fixed at 30. 30,000 is taken from arm 0.

⚠ **Data volume and optimiser budget move together here.** At 30 epochs, 5,000 pairs receives about
1,170 optimiser steps and 100,000 pairs about 23,460. This arm therefore measures *training-set size
under a fixed epoch budget* — a real quantity, and the one that describes how the model was actually
trained — but **not** an isolated effect of data volume. The step counts are printed below so the
confound stays in view. Arm T2 is the deconfounded version.

This replaces the old N-ablation, which was measured on the **AA-trained encoder of the
classifier-head architecture** and is therefore not a statement about the model in this thesis.

⚠ **100,000 is most of this arm's runtime.** Drop it from `TRAIN_GRID` first if the session is short.

In [ ]:
for n in TRAIN_GRID:
    if n == N_TRAIN: continue                  # arm 0 already has it
    for seed in SEEDS:
        run_config('T-train-size', lambda: EncPool(16), n, seed,
                   {EPOCHS: f'N={n:,}'}, tag=f'N={n:,}')

order = [f'N={n:,}' if n != N_TRAIN else 'K=16' for n in TRAIN_GRID]
idx   = [f'{n:,} pairs' for n in TRAIN_GRID]
print('=== Arm T - training set size at 30 epochs ===')
for m in ['spearman', 'spearman_high', 'auroc', 'map10', 'rmse_high']:
    table(['0-baseline', 'T-train-size'], m, order=order, index=idx)

d = _current(); d = d[d['arm'].isin(['0-baseline', 'T-train-size'])]
t = d.groupby('config')[['n_train', 'n_steps', 'train_mse_final']].mean().reindex(order)
t.index = idx
print('\n⚠ optimiser steps are NOT held fixed across these rows')
print(t.round(5).to_string())

## Arm T2 — training set size at matched optimiser steps *(optional)*

The same sizes, with the number of optimiser steps held at the baseline's by scaling the epoch count.
Any difference that survives here is attributable to the amount of data rather than to the amount of
optimisation, which is what a claim about data volume needs.

Each configuration costs about what the baseline costs, since the step count is the same. Enabled by
default; set `RUN_MATCHED_STEPS = False` in the constants cell to skip it.

Matched budget: **7,050 optimiser steps**, reached at 176 / 89 / 30 / 9 epochs for
5,000 / 10,000 / 30,000 / 100,000 pairs.

In [ ]:
if not RUN_MATCHED_STEPS:
    print('arm T2 disabled - set RUN_MATCHED_STEPS = True in the constants cell to run it')
    print(f'\n{"pairs":>10}{"steps/epoch":>14}{"epochs":>9}{"total steps":>14}')
    for n in TRAIN_GRID:
        e = EPOCHS if n == N_TRAIN else matched_epochs(n)
        print(f'{n:>10,}{steps_per_epoch(n):>14,}{e:>9}{steps_per_epoch(n)*e:>14,}')
else:
    for n in TRAIN_GRID:
        if n == N_TRAIN: continue              # the baseline already IS the matched budget
        e = matched_epochs(n)
        for seed in SEEDS:
            run_config('T2-matched-steps', lambda: EncPool(16), n, seed,
                       {e: f'N={n:,} matched'}, tag=f'N={n:,} @ {e} epochs')

    order2 = [f'N={n:,} matched' if n != N_TRAIN else 'K=16' for n in TRAIN_GRID]
    idx2   = [f'{n:,} pairs' for n in TRAIN_GRID]
    print('=== Arm T2 - training set size at matched optimiser steps ===')
    for m in ['spearman', 'spearman_high', 'auroc', 'map10', 'rmse_high']:
        table(['0-baseline', 'T2-matched-steps'], m, order=order2, index=idx2)

    d = _current(); d = d[d['arm'].isin(['0-baseline', 'T2-matched-steps'])]
    t = d.groupby('config')[['n_train', 'epochs', 'n_steps', 'train_mse_final']].mean().reindex(order2)
    t.index = idx2
    print('\nstep counts should now agree to within one epoch')
    print(t.round(5).to_string())

## Arm E — epoch budget

5 / 10 / 20 / 30 / 50 epochs at 30,000 pairs. **One training run per seed**, evaluated at each
checkpoint — see the note at the top of the notebook for why that is exactly equivalent to five
separate runs.

The 30-epoch row here is an independent second measurement of the deployed configuration, trained in
the same session. Its distance from arm 0 is a direct read on how much of the seed-to-seed movement in
Chapter 4 is session noise.

In [ ]:
for seed in SEEDS:
    run_config('E-epochs', lambda: EncPool(16), N_TRAIN, seed,
               {e: f'epochs={e}' for e in EPOCH_GRID}, tag='epoch sweep')

print('=== Arm E - epoch budget ===')
for m in ['spearman', 'spearman_high', 'auroc', 'map10', 'rmse_high']:
    table(['E-epochs'], m, order=[f'epochs={e}' for e in EPOCH_GRID])

d = _current(); d = d[d['arm'] == 'E-epochs']
print('\ntraining error by epoch budget (measured on the finished checkpoint)')
print(d.groupby('config')[['n_steps', 'train_loss_online', 'train_mse_final']].mean()
      .reindex([f'epochs={e}' for e in EPOCH_GRID]).round(5).to_string())

## Coverage check, tables and outputs

The output cell **refuses to write an incomplete experiment**. A complete run at the current protocol
signature is every configuration x 3 seeds x 4 datasets — 144 rows without arm T2, 180 with it. If
arms are still outstanding the cell lists them and stops; set `ALLOW_PARTIAL = True` to override,
which is right for an interim look and wrong for anything that reaches the chapter.

⚠ **AA prohibitions hold at source.** AA mid-range and high-range Spearman are written as empty by
construction, so they cannot be quoted by accident. AA MAP@10 and AUROC rest on 10 queries and 5
relevant pairs and never appear without that count.

In [ ]:
EXPECTED = ([('0-baseline', 'K=16')]
            + [('P-pooling-width', f'K={k}') for k in POOL_GRID if k != 16]
            + [('N-no-pooling', 'flatten')]
            + [('T-train-size', f'N={n:,}') for n in TRAIN_GRID if n != N_TRAIN]
            + [('E-epochs', f'epochs={e}') for e in EPOCH_GRID])
if RUN_MATCHED_STEPS:
    EXPECTED += [('T2-matched-steps', f'N={n:,} matched') for n in TRAIN_GRID if n != N_TRAIN]

RAW = _current().copy()
missing = [(a, c, s) for a, c in EXPECTED for s in SEEDS if not is_done(a, c, s)]
need = len(EXPECTED) * len(SEEDS) * len(DATASETS)

print(f'configurations expected: {len(EXPECTED)}   rows expected: {need}   rows present: {len(RAW)}')
if missing:
    print(f'\n⚠ {len(missing)} configuration/seed combinations are MISSING:')
    for a, c, s in missing: print(f'    {a:<18} {c:<18} seed {s}')
if not ALLOW_PARTIAL:
    assert not missing, (f'{len(missing)} configuration/seed combinations are missing - run the '
                         f'outstanding arms, or set ALLOW_PARTIAL = True for an interim look')
    assert len(RAW) == need, f'expected {need} rows at this protocol signature, found {len(RAW)}'
    print('\ncoverage complete')
else:
    print('\n⚠ ALLOW_PARTIAL is set - these outputs are interim and must not reach the chapter')

In [ ]:
KEYS = ['arm', 'config', 'dataset', 'n_train', 'epochs', 'n_steps', 'n_params']
NUM  = ['spearman', 'spearman_far', 'spearman_mid', 'spearman_high',
        'auroc', 'map10', 'rmse_high', 'train_loss_online', 'train_mse_final']
MEAN = RAW.groupby(KEYS, as_index=False)[NUM].mean()
SD   = (RAW.groupby(KEYS, as_index=False)[NUM].std(ddof=1)
        .rename(columns={c: c + '_sd' for c in NUM}))
MEAN = MEAN.merge(SD, on=KEYS)

RAW.to_csv('colab42_ablations_raw.csv', index=False)
MEAN.to_csv('colab42_ablations_mean.csv', index=False)

print('high-range Spearman across every configuration (seed means)')
t = MEAN.pivot_table(index='config', columns='dataset', values='spearman_high', aggfunc='mean')
print(t[[c for c in DATASETS if c in t.columns]].round(3).to_string())

summary = dict(
    protocol_sig=PROTOCOL_SIG,
    environment=ENV,
    evaluation_fingerprint=FINGERPRINT,
    config=dict(seeds=SEEDS, pool_grid=POOL_GRID, train_grid=TRAIN_GRID, epoch_grid=EPOCH_GRID,
                matched_steps=RUN_MATCHED_STEPS, allow_partial=ALLOW_PARTIAL,
                baseline=dict(K=16, n_train=N_TRAIN, epochs=EPOCHS, steps=BASE_STEPS),
                pair_seed=PAIR_SEED, syn_seed=SYN_SEED),
    evaluation_sizes={f: dict(pairs=int(len(STRAT[f]['nl'])),
                              high=int((STRAT[f]['nl'] >= RANGE_HIGH).sum()),
                              queries=len(REL[f]['T_high'])) for f in DATASETS},
    suppressed=sorted(f'{a}:{b}' for a, b in SUPPRESS_RANGE),
    chapter4_reference=CH4,
    results_mean=MEAN.round(6).to_dict('records'),
    results_by_seed=RAW.round(6).to_dict('records'))

with open('colab42_ablations.json', 'w') as fh: json.dump(summary, fh, indent=2, default=float)
json.loads(open('colab42_ablations.json').read())          # validate
print('\nwrote colab42_ablations.json')

In [ ]:
import shutil
OUTS = ['colab42_ablations_raw.csv', 'colab42_ablations_mean.csv', 'colab42_ablations.json']
for f in OUTS:
    shutil.copy2(f, CACHE); print('saved to Drive:', f)

# ⚠ Chrome blocks the second and later files of a multi-file download.
# If only the first arrives, take the rest from MyDrive/thesis_artefacts.
from google.colab import files
for f in OUTS: files.download(f)

## What to check before anything reaches Chapter 5

1. **The Stage A asserts all passed, including the subsample check.** Otherwise this run describes
   different data than Chapters 3 and 4 and none of it is comparable to them.
2. **Arm 0 sits within about 0.05 of the Chapter 4 row on every metric.** That is the whole basis for
   quoting the other arms next to Chapter 4's numbers. If it does not, nothing else here is usable.
3. **Coverage was complete** — `ALLOW_PARTIAL` was not used for the numbers that ship.
4. **Arm N's parameter count is quoted wherever arm N is discussed.** 1,648,512 against 141,184. The
   arm cannot separate pooling from capacity and the prose must say so.
5. **`K=32` is not described as reproducing the earlier 272,256-parameter result.** That one pooled
   over true length; this one pools over the padded width. Same parameter count, different model.
6. **Arm T is described as training-set size *at a fixed epoch budget*.** If the chapter makes a claim
   about how much data the model needs, it must rest on arm T2, where the optimiser budget is held
   fixed. Quote step counts either way.
7. **Only `train_mse_final` supports a statement about fitting the training set.**
   `train_loss_online` was accumulated while the parameters were still moving.
8. **Diminishing returns, never "plateau"**, for arms T and T2.
9. **No notebook names, file paths or Python identifiers reach the thesis.**